# ABCD Validation and SHAP Interpretation


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import shap
import joblib
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

RESULTS_DIR  = Path('results')
PICTURES_DIR = Path('pictures')
PICTURES_DIR.mkdir(exist_ok=True)

DPI         = 300
TARGETS     = ['ODI', 'SF36_PCS', 'SF36_MCS']
TARGET_LABELS = {'ODI': 'ODI (%)', 'SF36_PCS': 'SF-36 PCS', 'SF36_MCS': 'SF-36 MCS'}
MODEL_NAMES = ['Ridge', 'Lasso', 'ElasticNet', 'XGBoost', 'LightGBM', 'CatBoost']
MCID        = {'ODI': 10, 'SF36_PCS': 5, 'SF36_MCS': 5}

DATA_DIR = Path('data')

loocv_predictions  = joblib.load(RESULTS_DIR / 'loocv_predictions.pkl')
final_models       = joblib.load(RESULTS_DIR / 'final_models.pkl')
final_preprocessor = joblib.load(RESULTS_DIR / 'final_preprocessor.pkl')
feature_names      = joblib.load(RESULTS_DIR / 'feature_names.pkl')

X_raw    = pd.read_parquet(DATA_DIR / 'features.parquet')
X_all_sc = final_preprocessor.transform(X_raw)


In [2]:
def compute_abcd(y_true, y_pred, mcid):
    bias = float(np.mean(y_pred) - np.mean(y_true))
    slope, intercept, *_ = stats.linregress(y_pred, y_true)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    within_mcid = float(np.mean(np.abs(y_true - y_pred) <= mcid) * 100)
    return {
        'Bias (A)':   round(bias,        2),
        'β₀ (B)':     round(intercept,   3),
        'β₁ (B)':     round(slope,       3),
        'MAE (C)':    round(mae,          2),
        'RMSE (C)':   round(rmse,         2),
        'R² (C)':     round(r2,           3),
        '% MCID (D)': round(within_mcid,  1),
    }

records = []
for target_key in TARGETS:
    data   = loocv_predictions[target_key]
    y_true = data['y_true']
    for model_name in MODEL_NAMES:
        m = compute_abcd(y_true, data['y_pred'][model_name], MCID[target_key])
        records.append({'Target': target_key, 'Model': model_name, **m})

metrics_df = pd.DataFrame(records)
metrics_df


,Target,Model,Bias (A),β₀ (B),β₁ (B),MAE (C),RMSE (C),R² (C),% MCID (D)
0,ODI,Ridge,-0.03,0.589,0.952,5.50,7.06,0.320,88.0
1,ODI,Lasso,-0.04,0.491,0.962,5.48,7.04,0.325,87.7
2,ODI,ElasticNet,-0.03,0.468,0.963,5.52,7.07,0.319,88.2
3,ODI,XGBoost,-0.07,0.289,0.981,4.48,6.00,0.509,90.4
4,ODI,LightGBM,0.03,0.434,0.960,4.76,6.26,0.466,88.8
5,ODI,CatBoost,-0.01,-0.669,1.058,4.58,6.07,0.498,91.5
6,SF36_PCS,Ridge,0.01,-3.090,1.072,6.57,8.29,0.354,47.3
7,SF36_PCS,Lasso,0.01,2.014,0.953,6.70,8.37,0.341,45.3
8,SF36_PCS,ElasticNet,0.00,-2.604,1.060,6.62,8.39,0.339,47.3
9,SF36_PCS,XGBoost,-0.12,2.556,0.943,5.56,7.30,0.499,55.4


In [3]:
METRIC_COLS = ['Model', 'Bias (A)', 'β₀ (B)', 'β₁ (B)',
               'MAE (C)', 'RMSE (C)', 'R² (C)', '% MCID (D)']
ROW_H = 0.32
HDR_H = 0.48
FS    = 9

table_h = HDR_H + len(MODEL_NAMES) * ROW_H + 0.3
fig_h   = len(TARGETS) * table_h + 0.4
fig, axes = plt.subplots(len(TARGETS), 1, figsize=(14, fig_h), dpi=DPI)

for ax, target_key in zip(axes, TARGETS):
    df_t = (
        metrics_df[metrics_df['Target'] == target_key][METRIC_COLS]
        .reset_index(drop=True)
    )
    best_idx = df_t['MAE (C)'].astype(float).idxmin()
    ax.axis('off')

    tbl = ax.table(
        cellText=df_t.values.tolist(),
        colLabels=list(df_t.columns),
        cellLoc='center',
        loc='center',
        colWidths=[0.15, 0.11, 0.10, 0.10, 0.11, 0.11, 0.11, 0.12],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(FS)

    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#aaaaaa')
        cell.set_linewidth(0.5)
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold', fontsize=FS)
            cell.set_height(HDR_H / table_h)
        else:
            face = '#d5f5e3' if (row - 1) == best_idx else 'white'
            cell.set_facecolor(face)
            cell.set_height(ROW_H / table_h)

    ax.set_title(f'ABCD metrics: {TARGET_LABELS[target_key]}', fontsize=10, pad=8)

plt.tight_layout(pad=0.5)
out_path = PICTURES_DIR / 'table_metrics.png'
fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f'Saved: {out_path}')


Saved: pictures/table_metrics.png


In [4]:
_best_models = {}
for _tk in TARGETS:
    _y = loocv_predictions[_tk]['y_true']
    _best_models[_tk] = min(
        MODEL_NAMES,
        key=lambda mn: mean_absolute_error(_y, loocv_predictions[_tk]['y_pred'][mn])
    )

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=DPI)

for ax, target_key in zip(axes, TARGETS):
    model_name = _best_models[target_key]
    y_true = loocv_predictions[target_key]['y_true']
    y_pred = loocv_predictions[target_key]['y_pred'][model_name]

    ax.scatter(y_pred, y_true, s=10, alpha=0.4, color='#2980b9', linewidths=0)

    slope, intercept, *_ = stats.linregress(y_pred, y_true)
    xr = np.array([y_pred.min(), y_pred.max()])
    ax.plot(xr, intercept + slope * xr, color='#e74c3c', lw=1.8,
            label=f'β₁ = {slope:.3f},  β₀ = {intercept:.2f}')

    lo = min(y_pred.min(), y_true.min())
    hi = max(y_pred.max(), y_true.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.9, alpha=0.5, label='Ideal')

    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Observed',  fontsize=9)
    ax.set_title(
        f'{TARGET_LABELS[target_key]}  —  {model_name}\nMAE = {mae:.2f},  R² = {r2:.3f}',
        fontsize=10,
    )
    ax.legend(fontsize=8)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

fig.suptitle('Calibration plots: best models by MAE (LOOCV)', fontsize=12, y=1.02)
plt.tight_layout()

out_path = PICTURES_DIR / 'calibration_best_models.png'
fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f'Saved: {out_path}')


Saved: pictures/calibration_best_models.png


In [5]:
best_models = _best_models

print(f'Best models by MAE (LOOCV):')
print(f'  {"Target":<12} {"Model":<12} {"MAE":>7}  {"RMSE":>7}  {"R²":>7}  {"% MCID":>8}')
print('  ' + '-' * 58)

for target_key in TARGETS:
    data    = loocv_predictions[target_key]
    y_true  = data['y_true']
    best_mn = best_models[target_key]
    yp      = data['y_pred'][best_mn]
    mae  = mean_absolute_error(y_true, yp)
    rmse = root_mean_squared_error(y_true, yp)
    r2   = r2_score(y_true, yp)
    wm   = np.mean(np.abs(y_true - yp) <= MCID[target_key]) * 100
    print(f'  {target_key:<12} {best_mn:<12} {mae:7.3f}  {rmse:7.3f}  {r2:7.3f}  {wm:7.1f}%')


Best models by MAE (LOOCV):
  Target       Model            MAE     RMSE       R²    % MCID
  ----------------------------------------------------------
  ODI          XGBoost        4.476    5.999    0.509     90.4%
  SF36_PCS     CatBoost       5.342    6.869    0.557     55.4%
  SF36_MCS     XGBoost        5.980    7.945    0.405     52.5%


In [6]:
FEATURE_LABELS = {
    'age':                  'Age',
    'bmi':                  'BMI',
    'disease_duration':     'Disease duration',
    'back_pain_pre':        'Back pain (VAS)',
    'leg_pain_pre':         'Leg pain (VAS)',
    'sf36_phys_pre':        'SF-36 PCS (pre-op)',
    'sf36_psych_pre':       'SF-36 MCS (pre-op)',
    'odi_pre':              'ODI (pre-op)',
    'disc_height':          'Disc height',
    'sagittal_angle':       'Sagittal angle',
    'lumbar_lordosis':      'Lumbar lordosis',
    'linear_shift':         'Linear shift',
    'pfirrmann':            'Pfirrmann grade',
    'fujiwara':             'Fujiwara grade',
    'adc':                  'ADC',
    'facet_angle':          'Facet angle',
    'total_pain_pre':       'Total pain (VAS)',
    'leg_pain_proportion':  'Leg pain ratio',
    'back_pain_proportion': 'Back pain ratio',
    'onset_age':            'Age of onset',
    'degeneration_index':   'Degeneration index',
    'shift_height_ratio':   'Shift/height ratio',
    'adc_per_pfirrmann':    'ADC/Pfirrmann',
    'sf36_total_pre':       'SF-36 total (pre-op)',
}

OHE_LABELS = {
    'sex_female':              'Sex: female',
    'sex_male':                'Sex: male',
    'neuro_deficit_yes':       'Neurological deficit: yes',
    'neuro_deficit_no':        'Neurological deficit: no',
    'facet_tropism_present':   'Facet tropism: present',
    'facet_tropism_absent':    'Facet tropism: absent',
}

def get_label(fname):
    base = fname.split('__')[-1]
    return OHE_LABELS.get(base, FEATURE_LABELS.get(base, base))

display_names = [get_label(f) for f in feature_names]

shap_data = {}

for target_key in TARGETS:
    best_mn = best_models[target_key]
    model   = final_models[target_key][best_mn]

    if best_mn in ('XGBoost', 'LightGBM', 'CatBoost'):
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.LinearExplainer(model, X_all_sc)

    sv = explainer.shap_values(X_all_sc)

    mean_abs = np.abs(sv).mean(axis=0)
    top5     = np.argsort(mean_abs)[::-1][:5]

    shap_data[target_key] = {'shap_values': sv, 'model_name': best_mn}
    print(f'\n{target_key} ({best_mn}) — top-5 features by mean |SHAP|:')
    for idx in top5:
        print(f'  {display_names[idx]:<35s}  {mean_abs[idx]:.3f}')



ODI (XGBoost) — top-5 features by mean |SHAP|:
  ADC                                  2.180
  Sagittal angle                       0.916
  ADC/Pfirrmann                        0.860
  SF-36 PCS (pre-op)                   0.600
  Facet angle                          0.513

SF36_PCS (CatBoost) — top-5 features by mean |SHAP|:
  ADC                                  1.700
  Sagittal angle                       0.848
  ADC/Pfirrmann                        0.818
  Degeneration index                   0.773
  SF-36 MCS (pre-op)                   0.672

SF36_MCS (XGBoost) — top-5 features by mean |SHAP|:
  ADC/Pfirrmann                        1.628
  ADC                                  1.591
  SF-36 MCS (pre-op)                   1.195
  Total pain (VAS)                     0.772
  SF-36 PCS (pre-op)                   0.680


In [7]:
import io
from matplotlib.image import imread

panels = []
for target_key in TARGETS:
    sd = shap_data[target_key]
    shap.summary_plot(
        sd['shap_values'], X_all_sc,
        feature_names=display_names,
        max_display=20,
        show=False,
        plot_size=(7.5, 8),
    )
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close('all')
    buf.seek(0)
    panels.append(imread(buf))

fig, axes = plt.subplots(1, len(TARGETS), figsize=(24, 9), dpi=DPI)
for ax, target_key, img in zip(axes, TARGETS, panels):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(
        f'{TARGET_LABELS[target_key]} — {shap_data[target_key]["model_name"]}',
        fontsize=10, pad=8,
    )

plt.tight_layout()
out_path = PICTURES_DIR / 'shap_beeswarm.png'
fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f'Saved: {out_path}')


Saved: pictures/shap_beeswarm.png
